# Локальные языковые модели и структурированная генерация

Мы уже работали с моделями, которые преобразуют текст в векторное представление. Теперь перейдём к другому способу работы с текстом — **генеративным языковым моделям**.

В отличие от моделей эмбеддингов, которые возвращают векторное представление текста, генеративная языковая модель может формировать текстовый ответ на запрос пользователя.

Генеративную языковую модель можно использовать через внешний облачный сервис или запустить непосредственно на собственном компьютере или сервере.

Облачный вариант позволяет обращаться к большим и производительным моделям без собственного вычислительного оборудования. Однако передача данных во внешний сервис подходит не для всех задач: например, организация может ограничивать передачу конфиденциальной информации сторонним сервисам.

Локальный запуск позволяет контролировать, где обрабатываются данные и как настроено окружение, но требует собственных вычислительных ресурсов. Поэтому локальные модели обычно приходится выбирать с учётом доступной памяти и производительности, а их возможности могут уступать крупным облачным моделям.

Сначала кратко познакомимся с обращением к модели через облачный API, а затем перейдём к локальному запуску.

### Облачный доступ к языковой модели

Для первого эксперимента воспользуемся сервисом **OpenRouter** ([openrouter.ai](https://openrouter.ai/)), который предоставляет единый API для обращения к различным языковым моделям. Для небольшого эксперимента воспользуемся бесплатной моделью. Такой вариант удобен для знакомства с API, однако бесплатный доступ имеет ограничения, а доступные модели и лимиты могут меняться.


#### Настройка API-ключа

Для обращения к API потребуется персональный API-ключ. API-ключ является секретом, поэтому его не следует записывать непосредственно в исходный код или публиковать вместе с ноутбуком.

Для хранения конфигурационных параметров и секретов часто используют **переменные окружения**. В небольших проектах их удобно задавать в файле `.env` — текстовом файле, содержащем пары `ИМЯ=ЗНАЧЕНИЕ`. Специальные библиотеки могут загрузить эти значения в окружение программы, после чего приложение получает их через стандартные средства работы с переменными окружения.

Получите персональный API-ключ в OpenRouter и замените `ваш_ключ` в файле `.env` на полученный ключ:

```text
OPENROUTER_API_KEY=ваш_ключ
```

Для загрузки переменных из файла `.env` воспользуемся библиотекой `python-dotenv`.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv("/home/jovyan/.env")

api_key = os.getenv("OPENROUTER_API_KEY")

print(api_key is not None)

True


#### Первый запрос к облачной модели

Теперь обратимся к языковой модели через API.

Передадим модели простой текстовый запрос и выведем полученный ответ.

In [3]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

response = client.chat.completions.create(
    model="openai/gpt-4o",
    messages=[
        {
            "role": "user",
            "content": "Расскажи короткий анекдот про AI."
        }
    ],
    max_completion_tokens=200,
)

print(response.choices[0].message.content)

Конечно! Вот анекдот:

Почему искусственный интеллект никогда не бросает вызовы в шахматах?

Потому что он знает: с кем нянчиться, а у кого учиться!


Параметр `max_completion_tokens` задаёт верхнюю границу числа токенов, которые модель может потратить на генерацию ответа. Исторически для ограничения длины такого результата использовался параметр `max_tokens`. В современных версиях API `max_tokens` считается устаревшим именем (`deprecated`) и заменяется на `max_completion_tokens`. Новое название также точнее отражает работу современных моделей, для которых в лимит могут входить не только видимые токены ответа, но и токены внутреннего рассуждения.

Попробуем применить генерацию текста к задаче анализа предложения: попросим модель найти ошибки и предложить исправления.

In [3]:
sentence = """
Вчера учёный выступил с докладом о современных технолгиях которые
активно развиваются в последние годы, и отметил, что мы должны быть
благодарны Стиву Джобсу за создание Android.
"""

In [7]:
response = client.chat.completions.create(
    model="openai/gpt-4o",
    messages=[
        {
            "role": "user",
            "content": f"""
Найди ошибки в предложении, назови их тип и предложи вариант исправления.
{sentence}
"""
        }
    ],
    max_completion_tokens=200,
)

print(response.choices[0].message.content)

В этом предложении присутствуют несколько ошибок:

1. **Орфографическая ошибка**:
   - **Ошибка**: "технолгиях" 
   - **Тип ошибки**: Орфографическая ошибка — пропущена буква.
   - **Исправление**: "технологиях"

2. **Пунктуационная ошибка**:
   - **Ошибка**: Отсутствует запятая перед "которые".
   - **Тип ошибки**: Пунктуационная ошибка — сложноподчиненное предложение.
   - **Исправление**: "о современных технологиях, которые"

3. **Фактическая ошибка**:
   - **Ошибка**: "...мы должны быть благодарны Стиву Джобсу за создание Android."
   - **Тип ошибки**: Фактическая ошибка — андроид был создан компанией Google, а не Стивом Джобсом.
   - **Исправление**: "


### Переход к локальной модели

Облачный API позволяет быстро получить доступ к мощной языковой модели, но такой способ подходит не для всех задач. Если данные нельзя передавать во внешний сервис или требуется самостоятельно контролировать модель и окружение, языковую модель можно запустить локально.

В этом случае вычисления выполняются непосредственно в нашем окружении. Обратная сторона такого подхода — необходимость самостоятельно предоставить модели вычислительные ресурсы и выбрать модель, которая сможет на них работать.

Теперь попробуем тот же принцип работы с локальной моделью. В нашем Docker-окружении для этого используется отдельный сервис **Ollama**, который запускает модель и предоставляет API для обращения к ней.

В качестве локальной модели будем использовать `Qwen3:4B-Instruct` и посмотрим, как она справляется с различными типами задач.

#### Запрос к локальной модели

Ollama предоставляет API, совместимый с интерфейсом OpenAI. Поэтому для обращения к локальной модели можно использовать тот же подход, который мы уже применяли для облачного API. Отличаются адрес сервиса и название модели.

В отличие от облачного API, локальный Ollama не требует настоящего API-ключа. Однако библиотека `OpenAI` ожидает параметр `api_key`, поэтому при обращении к локальному сервису передадим условное значение.

In [7]:
from openai import OpenAI

client = OpenAI(
    base_url="http://ollama:11434/v1",
    api_key="ollama",
)

response = client.chat.completions.create(
    model="qwen3:4b-instruct",
    messages=[
        {
            "role": "user",
            "content": "Расскажи короткий анекдот про AI."
        }
    ],
    max_tokens=200,
)

print(response.choices[0].message.content)

Почему искусственный интеллект никогда не участвует в спорте?

Потому что он просто не умеет *играть* — он только *считает*! 😄


#### Локальная модель и время выполнения

Важно учитывать, что локальный и облачный примеры используют не только разные способы подключения, но и разные языковые модели. В облачном эксперименте мы обращались к `openai/gpt-4o` через `OpenRouter`, а здесь используем `Qwen3`. Поэтому различия в ответах нельзя связывать только со способом запуска: сами модели различаются размером, архитектурой и обучением.

Наша локальная модель содержит около 4 млрд параметров и используется в `Ollama` в квантованном формате `Q4_K_M`. Квантование позволяет хранить модель в более компактном виде и тем самым уменьшает требования к памяти по сравнению с использованием параметров в более точном представлении. Модель поддерживает контекст длиной до 262 144 токенов, а также работу с инструментами и режимы `thinking` и `completion`. В нашем эксперименте мы обращаемся к модели через `API Ollama` и используем её для генерации ответов на инструкции.

При выборе локальной модели приходится искать компромисс между качеством ответов, требованиями к вычислительным ресурсам и скоростью работы. Поэтому размер модели сам по себе не определяет её пригодность: важно проверить, насколько хорошо конкретная модель решает нужную задачу при доступных ресурсах и приемлемом времени выполнения.

#### Небольшое исследование возможностей локальной модели

Эти эксперименты не являются формальным тестом качества языковой модели. Они позволяют получить общее представление о том, как выбранная локальная модель справляется с разными типами задач: следованием инструкциям, извлечением информации, рассуждением, классификацией, работой с кодом и генерацией текста.

In [9]:
def ask_local_model(prompt):
    response = client.chat.completions.create(
        model="qwen3:4b-instruct",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=500,
    )
    
    return response.choices[0].message.content

In [11]:
print("Эксперимент 1. Следование инструкции\n")

prompt = """
Напиши три предложения о велосипеде.
Первое предложение должно быть вопросом,
второе предложение должно содержать число 7,
третье предложение должно состоять ровно из пяти слов.
"""

print(ask_local_model(prompt))

Эксперимент 1. Следование инструкции

А можно ли взять велосипед сегодня?  
Велосипед имеет семь основных частей.  
Велосипед весом, быстро, управляется.


In [12]:
print("Эксперимент 2. Извлечение информации и работа с датами\n")

prompt = """
27 февраля 2024 года Инна приехала в Казань. Через два дня она была уже в Самаре.
После этого в ближайшую субботу Инна планировала приехать в Уфу, но смогла добраться туда только на день позже.

Извлеки из текста информацию о городах и датах и представь её в виде списка событий.
"""

print(ask_local_model(prompt))

Эксперимент 2. Извлечение информации и работа с датами

Вот список событий, извлечённый из текста, с указанием городов и дат:

1. 27 февраля 2024 года — Инна приехала в Казань.  
2. 1 марта 2024 года — Инна была уже в Самаре (через два дня после прихода в Казань).  
3. В ближайшую субботу после посещения Самары — Инна планировала приехать в Уфу.  
4. 2 апреля 2024 года — Инна добралась в Уфу на день позже, чем планировала (то есть на день позже ближайшей субботы).

*Примечание:* Точная дата ближайшей субботы после 1 марта 2024 года — это 3 марта 2024 года. Значит, она добралась в Уфу 4 марта 2024 года.

Таким образом, уточнённый список событий:

- 27 февраля 2024 — Инна приехала в Казань.  
- 1 марта 2024 — Инна была в Самаре.  
- 3 марта 2024 — Инна планировала приехать в Уфу.  
- 4 марта 2024 — Инна добралась в Уфу.

**Итоговый список событий:**

1. 27 февраля 2024 — прибытие в Казань.  
2. 1 марта 2024 — пребывание в Самаре.  
3. 3 марта 2024 — планируемое прибытие в Уфу.  
4. 4 мар

In [13]:
print("Эксперимент 3. Логическое рассуждение\n")

prompt = """
У Лены, Максима и Оли разные домашние животные: кошка, собака и попугай.
Их любимые цвета также различаются: синий, зелёный и жёлтый.

Лена любит синий цвет. Максим держит собаку.
Хозяин попугая любит зелёный цвет. Хозяин кошки любит синий цвет.

Определи, какое животное и какой любимый цвет соответствуют каждому
человеку. Объясни ответ кратко, не более чем в трёх предложениях.
"""

print(ask_local_model(prompt))

Эксперимент 3. Логическое рассуждение

Лена любит синий цвет, а по условию хозяин кошки любит синий цвет, значит, у Лены — кошка.  
Максим держит собаку, значит, у него — собака.  
Остаётся Оля с попугаем, и по условию хозяин попугая любит зелёный цвет, значит, Оля — хозяйка попугая и любит зелёный цвет.  
Значит, у Лены — кошка и синий цвет, у Максима — собака (цвет не указан, значит — жёлтый), у Оли — попугай и зелёный цвет.


In [14]:
print("Эксперимент 4. Классификация тональности\n")

prompt = """
1. Доставка приехала даже раньше срока, всё отлично.
2. Товар пришёл вовремя, но упаковка была повреждена.
3. Заказ отменили без объяснения причины.
4. Цена высокая, зато качество отличное.
5. Доставка просто замечательная: всё перебили, ничего целым не приехало...

Определи тональность каждого отзыва: положительная, отрицательная
или смешанная, кратко поясни выбор.
"""

print(ask_local_model(prompt))

Эксперимент 4. Классификация тональности

1. **Доставка приехала даже раньше срока, всё отлично.**  
🔹 **Тональность: положительная**  
Пояснение: Упоминания о ранней доставке и "всё отлично" явно выражают удовлетворение.

2. **Товар пришёл вовремя, но упаковка была повреждена.**  
🔹 **Тональность: смешанная**  
Пояснение: Время доставки — положительный элемент, но повреждённая упаковка — негативное. Общая оценка неоднозначна.

3. **Заказ отменили без объяснения причины.**  
🔹 **Тонаыльность: отрицательная**  
Пояснение: Отсутствие объяснения при отмене заказа — неудобство и нарушение ожиданий, что передаёт негативную оценку.

4. **Цена высокая, зато качество отличное.**  
🔹 **Тональность: смешанная**  
Пояснение: Высокая цена — отрицательный аспект, но "качество отличное" — положительный. Ориентир на противоречивую оценку делает тон смешанным.

5. **Доставка просто замечательная: всё перебили, ничего целым не приехало...**  
🔹 **Тональность: отрицательная**  
Пояснение: Хотя «замечате

In [15]:
print("Эксперимент 5. Работа с кодом\n")

prompt = """
Что выведет этот код? Затем измени его так, чтобы он возвращал
квадраты отрицательных чисел.

numbers = [2, -5, 8, -11, 14]
result = [x * 2 for x in numbers if x % 2 == 0]
print(result)

Ответ дай кратко: сначала результат исходного кода,
затем исправленный код.
"""

print(ask_local_model(prompt))

Эксперимент 5. Работа с кодом

**Исходный код:**
```
[4, 10, 28]
```

**Исправленный код (квадраты отрицательных чисел):**
```python
numbers = [2, -5, 8, -11, 14]
result = [x ** 2 for x in numbers if x < 0]
print(result)
```


In [16]:
print("Эксперимент 6. Генерация с ограничениями\n")

prompt = """
Придумай короткую историю из пяти предложений.
В истории должны встретиться робот-пылесос, библиотека и апельсин,
причём каждый из этих объектов должен играть важную роль в сюжете.
"""

print(ask_local_model(prompt))

Эксперимент 6. Генерация с ограничениями

Робот-пылесос по имени Питер случайно включил старую архивную систему в библиотеке, где хранились древние рецепты. Вдруг он обнаружил рецепт волшебного апельсина — не просто фрукта, а источника позитивной энергии. Библиотекарь, увидев это, с смеющимся лицом предложил Питеру использовать апельсин, чтобы очистить и обогреть свою программу. Питер с удовольствием поднес апельсин к пыльной панели — и вдруг апельсин начал светиться, а библиотека вдруг раскрывалась как живой космический архив. Теперь каждый раз, когда Питер пылесосит, в комнате появляется легкий аромат апельсина — и в библиотеке появляется новая книга под названием «Секреты лёгких смехов».


#### Итоги экспериментов

Мы проверили несколько разных возможностей локальной модели: следование инструкциям, извлечение информации, рассуждение, классификацию, работу с кодом и генерацию текста. В каких задачах модель справилась уверенно, а где допустила ошибки или не выполнила часть инструкций?

Возможности языковой модели проявляются неравномерно: модель может хорошо справляться с одним типом задач и ошибаться в другом, в том числе в задачах, которые человеку кажутся простыми.

Важно учитывать, что такой небольшой набор примеров не позволяет формально оценить качество языковой модели. Его цель — получить первое наглядное представление о её возможностях и ограничениях.

### Управление формой ответа
При работе с языковой моделью важно формулировать не только содержание задачи, но и требования к форме ответа. В зависимости от задачи можно ограничивать объём ответа, задавать его структуру, указывать обязательные элементы и просить модель соблюдать определённый формат.

In [5]:
print(sentence)


Вчера учёный выступил с докладом о современных технолгиях которые
активно развиваются в последние годы, и отметил, что мы должны быть
благодарны Стиву Джобсу за создание Android.



In [10]:
print("Эксперимент 7.1 Свободный формат ответа\n")

prompt = f"""
{sentence}

Найди ошибки в предложении, назови их тип и предложи вариант исправления.
"""

print(ask_local_model(prompt))

Эксперимент 7.1 Свободный формат ответа

Предложение:

> «Вчера учёный выступил с докладом о современных технолгиях которые активно развиваются в последние годы, и отметил, что мы должны быть благодарны Стиву Джобсу за создание Android.»

---

### Ошибка 1: **Орфографическая ошибка**

**Форма слова**: *технолгии*  
**Ошибка**: написание слова "технологии" — пропущена буква **"о"**.

- **Правильно**: *технология* (не "технолгия")

✅ **Тип ошибки**: орфографическая (ошибка в написании слова)

❌ "технолгии" → ✅ "технологии"

---

### Ошибка 2: **Логическая и фактическая ошибка**

**Фраза**: *«мы должны быть благодарны Стиву Джобсу за создание Android»*

**Ошибка**: неверное утверждение о принадлежности Android.

- **Факт**: Android — это операционная система, разработанная **Лайонелом Бакером** и компанией **Google**, а не Стивом Джобсом.
- **Стив Джобс** — основатель Apple, создал **iOS** (операционную систему для iPhone), а не Android.

✅ **Тип ошибки**: логическая / фактическая ошибка 

#### Ограничения текстовых инструкций

Текстовая инструкция позволяет довольно гибко управлять формой ответа, однако она не гарантирует строгого соблюдения заданного формата.

Например, если попросить модель вернуть список в определённом виде, она может:

- изменить предложенную структуру;
- добавить пояснение до или после результата;
- использовать другие названия полей или категорий;
- добавить значения, которых не было среди предложенных вариантов;
- пропустить часть требуемых элементов;
- представить один и тот же тип данных в разных формах;
- вместо одного значения вернуть несколько или, наоборот, объединить несколько значений;
- сформировать данные, похожие на JSON, но добавить вокруг них обычный текст.

Причина в том, что в этом случае мы всё ещё просим модель **сгенерировать текст**, а не передаём ей формальное ограничение, которому результат обязан соответствовать.

Поэтому попробуем сделать инструкцию более строгой и явно зададим допустимые значения и структуру результата.

In [11]:
print("Эксперимент 7.2 Заданный формат ответа\n")

prompt = f"""
{sentence}

Найди ошибки в предложении и представь ответ кратко в виде списка:
фрагмент - тип ошибки - исправление.
Не давай дополнительных объяснений.
"""

print(ask_local_model(prompt))

Эксперимент 7.2 Заданный формат ответа

фрагмент — тип ошибки — исправление  
Вчера учёный выступил с докладом о современных технолгиях которые активно развиваются в последние годы, и отметил, что мы должны быть благодарны Стиву Джобсу за создание Android. — логическая ошибка — мы должны быть благодарны Стиву Джобсу за создание Android → мы должны быть благодарны Льву Бейкеру за создание Android  
Вчера учёный выступил с докладом о современных технолгиях которые активно развиваются в последние годы, и отметил, что мы должны быть благодарны Стиву Джобсу за создание Android. — орфографическая ошибка — технолгия → технология  
Вчера учёный выступил с докладом о современных технолгиях которые активно развиваются в последние годы, и отметил, что мы должны быть благодарны Стиву Джобсу за создание Android. — логическая ошибка — Стиву Джобсу за создание Android → Льву Бейкеру за создание Android


В предыдущем эксперименте мы задали форму ответа, но оставили модели свободу выбора самих категорий. Например, одну и ту же ошибку модель может назвать «фактической», «логической» или использовать другое описание.

При разовом использовании такой ответ может быть приемлемым. Однако если результат должен использоваться в дальнейшем — человеком без специальной подготовки или в автоматизированном процессе, — произвольные формулировки становятся проблемой. В первом случае человеку приходится самостоятельно разбираться в разных вариантах ответа, а во втором — дополнительно приводить их к единому виду и проверять допустимость полученных значений.

Поэтому попробуем дополнительно ограничить не только форму ответа, но и допустимые значения.

In [12]:
print("Эксперимент 7.3 Ограничение допустимых значений\n")

prompt = f"""
{sentence}

Найди ошибки в предложении и представь ответ кратко в виде списка:
фрагмент - тип ошибки - исправление.

Для типа ошибки используй только одно из следующих значений:
орфографическая, пунктуационная, грамматическая, лексическая, фактическая.

Не давай дополнительных объяснений.
"""

print(ask_local_model(prompt))

Эксперимент 7.3 Ограничение допустимых значений

- Вчера учёный выступил с докладом о современных технолгиях которые активно развиваются в последние годы, и отметил, что мы должны быть благодарны Стиву Джобсу за создание Android.  
  - технолгиях → технологиях (орфографическая)  
  - которые → что (грамматическая)  
  - благодарны → благодарны (орфографическая, но не требуется исправление)  
  - Стиву Джобсу → Стивену Джобсу (лексическая)  
  - за создание Android → за создание Android (фактическая)  

Кратко в виде списка:

- технолгиях — орфографическая — технологиях  
- которые — грамматическая — что  
- Стиву Джобсу — лексическая — Стивену Джобсу  
- за создание Android — фактическая — за создание iOS


#### Что изменилось после ограничения допустимых значений

В этом эксперименте мы добавили ещё одно ограничение: теперь модель должна была не только соблюдать форму ответа, но и использовать только заранее определённые категории ошибок.

В качестве категорий были выбраны:

- **орфографическая** — ошибка в написании слова;
- **пунктуационная** — ошибка в расстановке знаков препинания;
- **грамматическая** — нарушение грамматических правил, например неправильная форма слова или построение предложения;
- **лексическая** — неправильный выбор слова или нарушение его смыслового употребления;
- **фактическая** — ошибка в информации о реальном мире.

Эти категории выбраны так, чтобы охватить основные типы ошибок, которые могут встречаться в подобных текстах, и по возможности не пересекаться между собой.

Ограничение набора допустимых значений делает ответ более предсказуемым: вместо произвольных описаний модель должна выбирать из заранее заданного набора вариантов.

Однако результат всё ещё остаётся обычным текстом. Даже если модель использует нужные категории, она может добавить пояснения, изменить структуру ответа или представить информацию в другой форме. Попробуем сделать ещё один шаг и попросим модель представить результат в формате JSON.

In [15]:
print("Эксперимент 7.4 Ответ в формате JSON\n")

prompt = f"""
{sentence}

Найди ошибки в предложении.

Представь результат в формате JSON.
Для каждой ошибки укажи:
- фрагмент текста;
- тип ошибки;
- исправление.

Используй только следующие типы ошибок:
орфографическая, пунктуационная, грамматическая, лексическая, фактическая.

Не добавляй никаких пояснений до или после JSON.
"""

print(ask_local_model(prompt))

Эксперимент 7.4 Ответ в формате JSON

{
  "errors": [
    {
      "fragment": "технолгия",
      "type": "орфографическая",
      "correction": "технология"
    },
    {
      "fragment": "Стиву Джобсу за создание Android",
      "type": "фактическая",
      "correction": "Стиву Джобсу за создание iOS"
    }
  ]
}


#### Переход к структурированному результату

Переход к структурированному результату (в нашем случае — к JSON) позволяет представить результат в более структурированном виде. Вместо произвольного текста отдельные элементы ответа получают свои поля, в итоге результат становится проще хранить и анализировать.

Однако сама просьба вернуть JSON всё ещё не является строгим ограничением. Модель может:

- добавить пояснение до или после JSON;
- изменить названия полей;
- пропустить часть элементов;
- представить данные в другой структуре;
- использовать неожиданные значения.

Следующим шагом попробуем задать не только требование вернуть JSON, но и конкретную структуру ожидаемого результата.

In [14]:
print("Эксперимент 7.5 JSON заданной структуры\n")

prompt = f"""
{sentence}

Найди ошибки в предложении и представь результат в формате JSON.

Используй только следующие типы ошибок:
орфографическая, пунктуационная, грамматическая, лексическая, фактическая.

Ответ должен иметь следующую структуру:

{{
  "errors": [
    {{
      "fragment": "фрагмент текста с ошибкой",
      "type": "тип ошибки",
      "correction": "исправленный вариант"
    }}
  ]
}}

Не добавляй никаких пояснений до или после JSON.
"""

print(ask_local_model(prompt))

Эксперимент 7.5 JSON заданной структуры

{
  "errors": [
    {
      "fragment": "мы должны быть благодарны Стиву Джобсу за создание Android",
      "type": "фактическая",
      "correction": "мы должны быть благодарны Стиву Джобсу за создание iOS"
    },
    {
      "fragment": "вчера учёный выступил с докладом о современных технолгиях",
      "type": "орфографическая",
      "correction": "вчера учёный выступил с докладом о современных технологиях"
    }
  ]
}


### Ограничения формата и качество ответа

Однако у строгого формата есть и обратная сторона. Каждое новое ограничение добавляет модели дополнительное требование: нужно не только решить задачу, но и одновременно соблюдать все правила оформления ответа.

Если задача простая, это обычно не вызывает проблем. Но в более сложных случаях модель может начать нарушать одно из требований: например, дать менее полный ответ, ошибиться в содержании или всё равно отступить от заданной структуры.

Важно разделять две стороны качества результата: соответствие заданному формату и правильность самого ответа. Строгий формат помогает контролировать структуру результата, но сам по себе не гарантирует, что модель правильно решила задачу.

### Формальное задание структуры ответа (Structured Output)

До этого мы задавали требования к ответу с помощью текста запроса. Такой способ позволяет направлять модель, но результат всё равно остаётся обычным текстом. Если результат должен использоваться дальше — например, сохраняться в базе данных, передаваться другой программе или автоматически проверяться — важно зафиксировать, каким он должен быть.

Некоторые интерфейсы и API позволяют передавать структуру ответа отдельно от самого запроса. В этом случае мы задаём не только инструкцию, но и ожидаемую схему результата.

Например, можно указать:
- какие поля должны быть в ответе;
- какие типы данных они должны содержать;
- какие элементы не должны отсутствовать;
- какие дополнительные поля не допускаются.

Посмотрим, как это работает на простом примере.

In [55]:
response = client.chat.completions.create(
    model="qwen3:4b-instruct",
    messages=[
        {
            "role": "user",
            "content": """
            Инструмент с изящной рамой и множеством струн. Его музыка звучала сама по себе и усыпила трёхглавого пса Пушка. Что это?
            """
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "instrument_answer",
            "schema": {
                "type": "object",
                "properties": {
                    "instrument": {
                        "type": "string"
                    }
                },
                "required": [
                    "instrument"
                ],
                "additionalProperties": False
            }
        }
    }
)

print(response.choices[0].message.content)

{ "instrument": "арфа" }


В этом примере структура ответа передаётся отдельно от самого запроса. Мы не просто просим модель «ответить в формате JSON», а задаём схему ожидаемого результата.

Это важное отличие: теперь формат ответа становится отдельным параметром взаимодействия с моделью, а не только рекомендацией в тексте запроса. API получает схему отдельно и использует её при формировании ответа.

Это не означает, что модель получает какой-то новый «канал мышления» или второй скрытый контекст. Схема — это дополнительная информация для системы, которая управляет формированием ответа.

На уровне реализации возможны разные подходы:
- ограничение допустимых вариантов генерации во время построения ответа;
- проверка результата по схеме;
- комбинация этих подходов.

При этом схема описывает только форму результата. Она задаёт, какие поля должны быть в ответе и какие данные они должны содержать, но сама по себе не гарантирует, что модель правильно решила задачу.

Рассмотрим ещё один пример. Теперь результат будет не одним значением, а списком объектов с одинаковой структурой.

In [16]:
response = client.chat.completions.create(
    model="qwen3:4b-instruct",
    messages=[
        {
            "role": "user",
            "content": """
            Назови литературные произведения, в которых главным героем является ребёнок.
            Для каждого произведения укажи название и автора.
            """
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "children_heroes_books",
            "schema": {
                "type": "object",
                "properties": {
                    "works": {
                        "type": "array",
                        "minItems": 3,
                        "maxItems": 3,
                        "items": {
                            "type": "object",
                            "properties": {
                                "title": {
                                    "type": "string"
                                },
                                "author": {
                                    "type": "string"
                                }
                            },
                            "required": [
                                "title",
                                "author"
                            ],
                            "additionalProperties": False
                        }
                    }
                },
                "required": [
                    "works"
                ],
                "additionalProperties": False
            }
        }
    }
)

print(response.choices[0].message.content)

{ "works": [ { "author": "Александр Островский", "title": "Сказка о рыбаке и рыбке" }, { "author": "А.Н. Толстой", "title": "Капитан Воронцова" }, { "author": "Лев Толстой", "title": "Девочка у дуба" } ] }


Здесь результат имеет более сложную структуру: это список объектов, каждый из которых должен содержать одинаковый набор полей.

Схема может описывать не только поля объекта, но и дополнительные ограничения. Например, для списка можно задать минимальное или максимальное количество элементов с помощью параметров `minItems` и `maxItems`.

В этом примере мы задаём оба параметра равными `3`, поэтому схема требует ровно три элемента в списке. В отличие от предыдущего варианта, количество результатов теперь задаётся не в тексте запроса, а непосредственно в схеме ответа.

#### Описание структуры с помощью Pydantic

В предыдущих примерах схему ответа мы описывали вручную в виде JSON Schema. Для сложных структур такая схема быстро становится громоздкой: нужно явно указывать типы полей, вложенные объекты, списки и дополнительные ограничения.

**Pydantic** — отдельная Python-библиотека для описания и проверки структур данных с помощью моделей. Она не является частью Structured Output и сама по себе не задаёт модели формат ответа. В нашем случае мы используем Pydantic как удобный способ описать структуру данных, а затем получить из этой модели JSON Schema, которую можно передать API для Structured Output.

Например, элемент списка можно представить как объект `Work`, а весь ответ — как список таких объектов:

```python
class Work(BaseModel):
    title: str
    author: str

class Works(BaseModel):
    works: list[Work]
```

Дополнительные ограничения также можно задавать в модели. Например, можно указать, что список должен содержать ровно три элемента: `Field(min_length=3, max_length=3)`.

При этом меняется только способ описания структуры. Сама идея остаётся прежней: мы задаём ожидаемый формат результата отдельно от текста задачи.

In [60]:
from pydantic import BaseModel, Field

class Work(BaseModel):
    title: str
    author: str

class Works(BaseModel):
    works: list[Work] = Field(min_length=3, max_length=3)


response = client.chat.completions.create(
    model="qwen3:4b-instruct",
    messages=[
        {
            "role": "user",
            "content": """
            Назови литературные произведения, в которых главным героем является ребёнок.
            Для каждого произведения укажи название и автора.
            """
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "works",
            "schema": Works.model_json_schema()
        }
    }
)

result = Works.model_validate_json(
    response.choices[0].message.content
)

print(result)

works=[Work(title='The Little Prince', author='Antoine de Saint-Exupéry'), Work(title='Where the Wild Things Are', author='Maurice Sendak'), Work(title='Matilda', author='Roald Dahl')]


#### Использование Pydantic для описания схемы и обработки результата

В этом примере Pydantic использовался на двух этапах.

Сначала модель `Works` была преобразована в JSON Schema с помощью `Works.model_json_schema()`.

Эта схема была передана в `response_format` и использовалась при формировании ответа модели.

После получения ответа JSON был преобразован обратно в объект Pydantic с помощью `Works.model_validate_json(response.choices[0].message.content)`.

Таким образом, одна и та же модель Python используется для описания ожидаемой структуры и для обработки полученного результата.

Теперь применим этот подход к нашей задаче о поиске ошибок.

In [17]:
from typing import Literal
from pydantic import BaseModel


class Error(BaseModel):
    fragment: str
    type: Literal[
        "орфографическая",
        "пунктуационная",
        "грамматическая",
        "лексическая",
        "фактическая"
    ]
    correction: str


class ErrorAnalysis(BaseModel):
    errors: list[Error]


response = client.chat.completions.create(
    model="qwen3:4b-instruct",
    messages=[
        {
            "role": "user",
            "content": f"""
{sentence}

Найди ошибки в предложении и представь результат в заданном формате.
Для каждой ошибки укажи фрагмент текста, тип ошибки и исправление.
"""
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "error_analysis",
            "schema": ErrorAnalysis.model_json_schema()
        }
    }
)


result = ErrorAnalysis.model_validate_json(
    response.choices[0].message.content
)

print(result)

errors=[Error(fragment='Вчера учёный выступил с докладом о современных технолгиях которые активно развиваются в последние годы', type='орфографическая', correction='Вчера учёный выступил с докладом о современных технолгиях которые активно развиваются в последние годы'), Error(fragment='и отметил, что мы должны быть благодарны Стиву Джобсу за создание Android', type='фактическая', correction='и отметил, что мы должны быть благодарны Стиву Джобсу за создание Android')]


### Структура ответа и проверка качества

Structured Output и Pydantic позволяют значительно лучше контролировать форму ответа. Теперь мы можем заранее задать:

- какие поля должны присутствовать в результате;
- какие типы данных должны находиться в этих полях;
- какие значения допустимы для отдельных полей.

Однако это ограничивает только структуру ответа, а не гарантирует правильность самого решения задачи.

Например, модель может вернуть полностью корректный объект:

- поле `fragment` будет содержать текстовый фрагмент;
- поле `type` будет одним из допустимых типов ошибок;
- поле `correction` будет строкой;

но при этом сама классификация может быть неверной, исправление может быть ошибочным или модель может пропустить часть ошибок.

То есть после введения схемы появляются две разные задачи проверки:

1. **Проверка формата** — соответствует ли результат заданной структуре.
2. **Проверка содержания** — правильно ли модель решила исходную задачу.

До появления структурированного вывода эти проверки было сложно разделить: результат приходилось сначала вручную разбирать, чтобы понять, где заканчивается ответ модели и где начинается сама информация.

Теперь структура ответа фиксирована, поэтому содержательную часть результата можно проверять отдельно.

Например, для задачи поиска ошибок можно заранее подготовить предложения с известными ошибками и проверить:

- нашла ли модель все ошибки;
- не добавила ли лишние ошибки;
- правильно ли определила тип;
- предложила ли корректное исправление.

Таким образом, структурированный вывод не заменяет проверку качества, но позволяет отделить проверку структуры от проверки содержания.

#### Сравнение с эталонным результатом

Так как теперь ответ имеет фиксированную структуру, его можно не только читать вручную, но и сравнивать с заранее подготовленным эталоном.

Для простого учебного примера будем использовать небольшой набор предложений с известными ошибками. Для каждого предложения заранее зададим ожидаемый результат и проверим, совпадает ли ответ модели с эталоном.

В реальных задачах такое сравнение обычно требует более сложных правил. Например, возможны ситуации, когда:

- модель найдёт ошибку, которую не учёл составитель эталона;
- модель предложит другой, но допустимый вариант исправления;
- два варианта формулировки будут иметь одинаковый смысл;
- часть ошибок будет пропущена.

Поэтому проверка может быть как строгой (требовать полного совпадения), так и более гибкой (проверять только отдельные свойства результата).

In [62]:
test_cases = [
    {
        "text": "Он написал доклад о современных технолгиях.",
        "expected": [
            {
                "fragment": "технолгиях",
                "type": "орфографическая",
                "correction": "технологиях"
            }
        ]
    },
    {
        "text": "Мы благодарны Стиву Джобсу за создание Android.",
        "expected": [
            {
                "fragment": "создание Android",
                "type": "фактическая",
                "correction": "создание iPhone"
            }
        ]
    },
    {
        "text": "Учёный отметил что технологии быстро развиваются.",
        "expected": [
            {
                "fragment": "отметил что технологии",
                "type": "пунктуационная",
                "correction": "отметил, что технологии"
            }
        ]
    },
    {
        "text": """
        Вчера учёный выступил с докладом о современных технолгиях которые
        активно развиваются в последние годы, и отметил, что мы должны быть
        благодарны Стиву Джобсу за создание Android.
        """,
        "expected": [
            {
                "fragment": "технолгиях",
                "type": "орфографическая",
                "correction": "технологиях"
            },
            {
                "fragment": "технолгиях которые",
                "type": "пунктуационная",
                "correction": "технологиях, которые"
            },
            {
                "fragment": "создание Android",
                "type": "фактическая",
                "correction": "создание iPhone"
            }
        ]
    }
]

In [65]:
from collections import Counter


for case in test_cases:
    response = client.chat.completions.create(
        model="qwen3:4b-instruct",
        messages=[
            {
                "role": "user",
                "content": f"""
                {case["text"]}

                Найди ошибки в предложении и представь результат в заданном формате.

                Для каждой ошибки:
                - укажи только короткий фрагмент текста с ошибкой;
                - укажи тип ошибки;
                - укажи исправление только для этого фрагмента, а не для всего предложения.

                Не переписывай предложение целиком.
                """
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "error_analysis",
                "schema": ErrorAnalysis.model_json_schema()
            }
        }
    )

    result = ErrorAnalysis.model_validate_json(
        response.choices[0].message.content
    )

    found_types = Counter(error.type for error in result.errors)
    expected_types = Counter(item["type"] for item in case["expected"])

    print("Текст:")
    print(case["text"].strip())

    print("\nОжидались:")
    for error_type, count in expected_types.items():
        print(f"  {error_type}: {count}")

    print("Найдены:")
    for error_type, count in found_types.items():
        print(f"  {error_type}: {count}")

    missing = expected_types - found_types
    extra = found_types - expected_types

    if missing:
        print("Пропущены:")
        for error_type, count in missing.items():
            print(f"  {error_type}: {count}")
    else:
        print("Пропущены: нет")

    if extra:
        print("Лишние:")
        for error_type, count in extra.items():
            print(f"  {error_type}: {count}")
    else:
        print("Лишние: нет")

    print()

Текст:
Он написал доклад о современных технолгиях.

Ожидались:
  орфографическая: 1
Найдены:
  орфографическая: 1
Пропущены: нет
Лишние: нет

Текст:
Мы благодарны Стиву Джобсу за создание Android.

Ожидались:
  фактическая: 1
Найдены:
  фактическая: 1
Пропущены: нет
Лишние: нет

Текст:
Учёный отметил что технологии быстро развиваются.

Ожидались:
  пунктуационная: 1
Найдены:
Пропущены:
  пунктуационная: 1
Лишние: нет

Текст:
Вчера учёный выступил с докладом о современных технолгиях которые
        активно развиваются в последние годы, и отметил, что мы должны быть
        благодарны Стиву Джобсу за создание Android.

Ожидались:
  орфографическая: 1
  пунктуационная: 1
  фактическая: 1
Найдены:
  фактическая: 1
Пропущены:
  орфографическая: 1
  пунктуационная: 1
Лишние: нет



#### Возможности и ограничения автоматической проверки

Получив структурированный результат, мы можем не только вывести ответ модели, но и сравнить его с заранее подготовленным эталоном.

В простом примере мы сравниваем только количество и типы найденных ошибок. Это уже позволяет автоматически обнаружить некоторые проблемы:

- модель пропустила ожидаемую ошибку;
- модель добавила ошибку другого типа;
- количество найденных ошибок отличается от ожидаемого.

Однако такой способ сравнения является только одним из возможных вариантов. В зависимости от задачи проверку можно сделать более строгой или более гибкой.

Например, можно дополнительно учитывать:

- соответствие найденного фрагмента исходному тексту;
- правильность предложенного исправления;
- положение ошибки в предложении;
- допустимые варианты исправления одной и той же ошибки;
- случаи, когда модель нашла ошибку, отсутствующую в эталоне.

При этом возникает отдельная проблема: эталон сам не всегда является абсолютной истиной. В задачах с языком возможны разные корректные исправления или спорные случаи классификации. Поэтому критерии сравнения нужно выбирать под конкретную задачу.

Структурированный вывод не решает автоматически проблему оценки качества, но делает такие проверки возможными: результат имеет явные поля, которые можно анализировать программно.

#### Настройка промпта и проверка качества результата

Даже при использовании структурированного вывода текстовая инструкция остаётся важной частью запроса. Однако добавление новых требований не всегда улучшает результат.

Каждое дополнительное указание становится ещё одной задачей, которую модель должна учитывать одновременно с решением основной задачи. Например, ей нужно не только найти ошибки, но и:

- выбрать правильные фрагменты текста;
- определить тип ошибки;
- предложить исправление;
- соблюдать ограничения формата ответа.

Если инструкция становится слишком длинной и содержит много деталей, особенно для небольших моделей, часть требований может быть выполнена хуже или проигнорирована.

Поэтому не стоит пытаться описать в промпте всё возможное. Лучше разделять задачи:

- то, что относится к структуре данных, задавать через схему;
- то, что относится к смыслу задачи, оставлять в инструкции;
- добавлять только те текстовые ограничения, которые действительно помогают получить нужный результат.

В нашем случае схема уже задаёт допустимые поля и значения, поэтому в промпте достаточно уточнить особенности, которые нельзя выразить структурой: например, что в поле `fragment` нужно указывать короткий фрагмент исходного текста, а в `correction` — только исправление этого фрагмента, а не переписывать всё предложение.

Наличие структурированного результата позволяет систематически тестировать разные варианты промпта, схемы и модели на одном и том же наборе примеров.

При этом важно понимать, что сама проверка результата тоже является отдельной задачей. Простое сравнение строк часто бывает слишком строгим: модель может найти ту же ошибку, но выделить немного другой фрагмент текста или предложить другой допустимый вариант исправления.

Поэтому критерии проверки нужно выбирать в зависимости от цели эксперимента.

Таким образом, структурированный вывод не делает модель автоматически правильной, но создаёт основу для контроля качества: результат становится достаточно формальным, чтобы его можно было анализировать, сравнивать и использовать для экспериментов.